In [22]:
import numpy as np
import pandas as pd
import random
import time
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import rankdata
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.metrics import pairwise_distances

# --- 1. CHAOTIC DATA GENERATION (Simulating Nanopore Noise) ---
print("--- Generating 1000 Biological Entities ---")
random.seed(42)
np.random.seed(42)

def mutate_sequence(seq, error_rate=0.05):
    """Simulates Nanopore noise: Indels (fatal for one-hot) and Subs."""
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            type_r = random.random()
            if type_r < 0.33: new_seq.append(random.choice("ACGT")) # Sub
            elif type_r < 0.66: pass # Del (Frame shift!)
            else: 
                new_seq.append(base)
                new_seq.append(random.choice("ACGT")) # Ins
        else:
            new_seq.append(base)
    return "".join(new_seq)

def generate_dna(length): 
    return "".join(random.choices("ACGT", k=length))

# Generate True Entities
n_items = 1000
pool_barcodes = [generate_dna(20) for _ in range(n_items)]
pool_inserts  = [generate_dna(150) for _ in range(n_items)]

# Define Truth & Traps
combos = []
# Trap 1: Same Barcode, Different Insert (Should be split)
combos.append({"Label": "Trap_SharedBC_A", "BC": pool_barcodes[0], "Ins": pool_inserts[0]})
combos.append({"Label": "Trap_SharedBC_B", "BC": pool_barcodes[0], "Ins": pool_inserts[1]})
# Trap 2: Different Barcode, Same Insert (Should be split)
combos.append({"Label": "Trap_SharedIns_A", "BC": pool_barcodes[1], "Ins": pool_inserts[2]})
combos.append({"Label": "Trap_SharedIns_B", "BC": pool_barcodes[2], "Ins": pool_inserts[2]})

for i in range(3, n_items):
    combos.append({"Label": f"Standard_{i}", "BC": pool_barcodes[i], "Ins": pool_inserts[i]})

# Generate Reads (1-3 reads per entity)
data = []
for combo in combos:
    n_reads = random.choices([4, 6, 8], weights=[0.2, 0.4, 0.4])[0]
    for _ in range(n_reads):
        data.append({
            "Label": combo["Label"], 
            "Barcode": mutate_sequence(combo["BC"]), 
            "Insert": mutate_sequence(combo["Ins"])
        })

df = pd.DataFrame(data)
print(f"Dataset Generated: {len(df)} reads.")


# --- 2. BARCODE VECTORIZATION (The Short K-mer Fix) ---
print("\n[1/4] Processing Barcodes (Shift-Invariant K-mers)...")

# CRITICAL: For short sequences (15-20nt) with indels:
# 1. ngram_range=(2, 3): Captures Bigrams (robust) AND Trigrams (specific).
# 2. No SVD: The feature space is small enough (~300 dims), SVD loses too much info here.
bc_vectorizer = CountVectorizer(
    analyzer='char', 
    ngram_range=(2, 3), # <--- The "Short K-mer" Logic
    binary=False        # Count frequency (e.g., 'AA' appearing twice matters in short seqs)
)

bc_vectors = bc_vectorizer.fit_transform(df['Barcode'])

# Normalize: Critical because read lengths vary due to indels
normalizer = Normalizer(norm='l2')
bc_vectors = normalizer.transform(bc_vectors.astype(float))


# --- 3. INSERT VECTORIZATION (Standard SVD) ---
print("[2/4] Processing Inserts (LSA/SVD)...")

# Inserts are long (150bp). We use larger K (4) and SVD to reduce noise.
ins_vectorizer = CountVectorizer(analyzer='char', ngram_range=(4, 4), binary=False)
svd = TruncatedSVD(n_components=50, random_state=42)

ins_raw = ins_vectorizer.fit_transform(df['Insert'])
ins_vectors = normalizer.transform(svd.fit_transform(ins_raw))


# --- 4. ROBUST FUSION (Rank Norm + Maximum) ---
print("[3/4] Fusing Signals...")

# Calculate Raw Cosine Distances
dist_bc = pairwise_distances(bc_vectors, metric='cosine')
dist_ins = pairwise_distances(ins_vectors, metric='cosine')

def rank_normalize(matrix):
    """
    Converts distances to Percentiles (0.0 to 1.0).
    Solves the issue where Barcode distance 0.2 != Insert distance 0.2
    """
    # rankdata flattens the array, ranks them, then we reshape back
    ranked = rankdata(matrix)
    return (ranked.reshape(matrix.shape) - 1) / (ranked.max() - 1)

# Apply Rank Normalization
norm_bc = rank_normalize(dist_bc)
norm_ins = rank_normalize(dist_ins)

# Weighting: We trust Barcodes slightly more (1.0 vs 0.9)
norm_bc = norm_bc * 1.0
norm_ins = norm_ins * 0.9 

# STRICT FUSION: Use Maximum
# If Barcodes say "Different" (1.0) but Inserts say "Same" (0.0) -> Result is "Different" (1.0)
final_matrix = np.maximum(norm_bc, norm_ins)

np.fill_diagonal(final_matrix, 0)


# --- 5. CLUSTERING & AUTO-TUNING ---
print(f"[4/4] Auto-Tuning Clusters (Target: ~{n_items})...")

condensed_matrix = squareform(final_matrix)
Z = linkage(condensed_matrix, method='average')

# Stability Analysis
merge_distances = Z[:, 2]
n_samples = final_matrix.shape[0]
target_k = n_items
window = 200 # Search window

best_k = target_k
max_lifetime = -1.0
best_threshold = 0.0

start_k = max(2, target_k - window)
end_k = min(n_samples - 1, target_k + window)

for k in range(start_k, end_k):
    idx = n_samples - k
    if idx >= len(merge_distances): continue
    
    # "Lifetime" is how much distance grows before the next merge happens
    lifetime = merge_distances[idx] - merge_distances[idx-1]
    
    if lifetime > max_lifetime:
        max_lifetime = lifetime
        best_k = k
        # Cut in the middle of the stable zone
        best_threshold = merge_distances[idx-1] + (lifetime / 2)

print(f"   Stability Optimized K: {best_k} | Threshold: {best_threshold:.4f}")

# Apply Cluster Labels
labels = fcluster(Z, t=best_threshold, criterion='distance')
df['Cluster'] = labels


# --- 6. VALIDATION ---
print("\n--- Final Forensic Report ---")

# Check Trap 1: Same Barcode, Different Insert
t1 = df[df['Label'].str.contains("Trap_SharedBC")]
t1_clusters = t1['Cluster'].unique()
print(f"Trap 1 (Shared BC, Diff Ins): Found in {len(t1_clusters)} clusters (Expected 2).")
if len(t1_clusters) == 2: print("   ✅ SUCCESS: Inserts forced a split.")
else: print("   ❌ FAIL: Merged incorrectly.")

# Check Trap 2: Different Barcode, Same Insert
t2 = df[df['Label'].str.contains("Trap_SharedIns")]
t2_clusters = t2['Cluster'].unique()
print(f"Trap 2 (Diff BC, Shared Ins): Found in {len(t2_clusters)} clusters (Expected 2).")
if len(t2_clusters) == 2: print("   ✅ SUCCESS: Barcodes forced a split.")
else: print("   ❌ FAIL: Merged incorrectly.")

# Global fragmentation check
n_final = len(set(df['Cluster']))
print(f"\nTotal Clusters Found: {n_final}")
print(f"Total True Entities:  {len(combos)}")
accuracy = 1.0 - (abs(n_final - len(combos)) / len(combos))
print(f"Estimated Accuracy:   {accuracy*100:.1f}%")

--- Generating 1000 Biological Entities ---
Dataset Generated: 6404 reads.

[1/4] Processing Barcodes (Shift-Invariant K-mers)...
[2/4] Processing Inserts (LSA/SVD)...
[3/4] Fusing Signals...
[4/4] Auto-Tuning Clusters (Target: ~1000)...
   Stability Optimized K: 1000 | Threshold: 0.0300

--- Final Forensic Report ---
Trap 1 (Shared BC, Diff Ins): Found in 2 clusters (Expected 2).
   ✅ SUCCESS: Inserts forced a split.
Trap 2 (Diff BC, Shared Ins): Found in 2 clusters (Expected 2).
   ✅ SUCCESS: Barcodes forced a split.

Total Clusters Found: 1000
Total True Entities:  1001
Estimated Accuracy:   99.9%



--- 🔬 Forensic Analysis of Decoys (Traps) ---

[Trap 1] Shared Barcode Test:
   Reads involved: 5
   Assigned to Clusters: [ 80 493]
   ✅ SUCCESS: The algorithm forced a split based on Insert differences.
      Split Balance: 3 reads vs 2 reads

[Trap 2] Shared Insert Test:
   Reads involved: 5
   Assigned to Clusters: [431 111]
   ✅ SUCCESS: The algorithm forced a split based on Barcode differences.

[Control] Purity Check:
   Standard_227: ✅ Stable
   Standard_76: ✅ Stable
   Standard_497: ✅ Stable
   Standard_776: ✅ Stable
   Standard_429: ✅ Stable

🏆 RESULT: ALGORITHM IS VALIDATED.
   It successfully balanced Barcode and Insert signals to distinguish all biological entities.


In [2]:
import numpy as np
import pandas as pd
import random
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import pairwise_distances_chunked

# --- 1. CHAOTIC DATA GENERATION (Unchanged) ---
print("--- Generating Simulated Data (Targeting Scalability) ---")
random.seed(42)
np.random.seed(42)

def mutate_sequence(seq, error_rate=0.05):
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            type_r = random.random()
            if type_r < 0.33: new_seq.append(random.choice("ACGT")) 
            elif type_r < 0.66: pass 
            else: 
                new_seq.append(base)
                new_seq.append(random.choice("ACGT")) 
        else:
            new_seq.append(base)
    return "".join(new_seq)

def generate_dna(length): 
    return "".join(random.choices("ACGT", k=length))

# Let's bump this up to test scalability (try 2000-5000 first on your machine)
n_entities = 2000 
print(f"Simulating {n_entities} entities...")

pool_barcodes = [generate_dna(20) for _ in range(n_entities)]
pool_inserts  = [generate_dna(150) for _ in range(n_entities)]

combos = []
# Trap 1: Same BC, Diff Ins
combos.append({"Label": "Trap_SharedBC_A", "BC": pool_barcodes[0], "Ins": pool_inserts[0]})
combos.append({"Label": "Trap_SharedBC_B", "BC": pool_barcodes[0], "Ins": pool_inserts[1]})
# Trap 2: Diff BC, Same Ins
combos.append({"Label": "Trap_SharedIns_A", "BC": pool_barcodes[1], "Ins": pool_inserts[2]})
combos.append({"Label": "Trap_SharedIns_B", "BC": pool_barcodes[2], "Ins": pool_inserts[2]})

for i in range(3, n_entities):
    combos.append({"Label": f"Standard_{i}", "BC": pool_barcodes[i], "Ins": pool_inserts[i]})

data = []
for combo in combos:
    n_reads = random.choices([4, 6], weights=[0.5, 0.5])[0]
    for _ in range(n_reads):
        data.append({
            "Label": combo["Label"], 
            "Barcode": mutate_sequence(combo["BC"]), 
            "Insert": mutate_sequence(combo["Ins"])
        })

df = pd.DataFrame(data)
print(f"Dataset Generated: {len(df)} reads.")


# --- 2. VECTORIZATION (Unchanged) ---
print("\n[1/3] Vectorizing Sequences...")

# Barcodes: Short K-mers
bc_vectorizer = CountVectorizer(analyzer='char', ngram_range=(2, 3), binary=False)
bc_vectors = bc_vectorizer.fit_transform(df['Barcode'])
normalizer = Normalizer(norm='l2')
bc_vectors = normalizer.transform(bc_vectors.astype(np.float32)) # Use float32 to save RAM

# Inserts: SVD
ins_vectorizer = CountVectorizer(analyzer='char', ngram_range=(6, 6), binary=False)
svd = TruncatedSVD(n_components=50, random_state=42)
ins_raw = ins_vectorizer.fit_transform(df['Insert'])
ins_vectors = normalizer.transform(svd.fit_transform(ins_raw)).astype(np.float32)


# --- 3. SCALABLE FUSION (The Fix) ---
print("[2/3] Fusing Signals (Sparse/Gated Calculation)...")

# We need to build the "Condensed Distance Matrix" (1D array) directly.
# This avoids ever creating the N*N square matrix.
# Size of condensed matrix = N * (N-1) / 2
n_samples = bc_vectors.shape[0]
n_condensed = n_samples * (n_samples - 1) // 2
condensed_matrix = np.ones(n_condensed, dtype=np.float32) # Init with 1.0 (Max Distance)

# Logic Parameters
BC_GATE_THRESHOLD = 0.6  # Only look at inserts if BC dist < 0.6
CHUNK_SIZE = 1000        # Process rows in chunks to manage RAM

# Helper: Convert 2D indices (i, j) where j > i to 1D condensed index
# Note: Doing this per-pixel is slow, so we iterate sequentially to match condensed order.

k_counter = 0 # Points to current position in condensed_matrix
print(f"    Target condensed size: {n_condensed:,} elements.")

# We iterate over the matrix in the exact order 'pdist' would:
# Row 0 vs 1..N
# Row 1 vs 2..N
# ...
for i in range(n_samples - 1):
    # Log progress every 10%
    if i % max(1, (n_samples // 10)) == 0:
        print(f"    Processing Row {i}/{n_samples}...")
        
    # Get the vector for the current row i
    row_bc = bc_vectors[i]
    row_ins = ins_vectors[i]
    
    # We need to compare row i against all rows j > i
    # Slicing is efficient in CSR matrices
    block_bc = bc_vectors[i+1:]
    block_ins = ins_vectors[i+1:]
    
    # 1. Calculate Barcode Distances for this block (Vectorized)
    # Cosine Dist = 1 - Dot Product (since vectors are L2 normalized)
    # dense=True is safe here because block is (N-i) x 1, which is small.
    dots_bc = block_bc.dot(row_bc.T).toarray().flatten()
    dists_bc = 1.0 - dots_bc
    dists_bc[dists_bc < 0] = 0 # Float precision safety
    
    # 2. Identify candidates (The Gating Logic)
    # Which indices in this block pass the "vaguely similar" check?
    mask_candidates = dists_bc < BC_GATE_THRESHOLD
    
    # 3. Initialize Fusion Distances with Barcode Distances
    # (If we skip insert, we default to the "bad" barcode distance or 1.0)
    # Here we default to 1.0 for non-candidates to force separation
    fused_dists = np.ones_like(dists_bc) 
    
    # 4. Calculate Insert Distances ONLY for candidates
    if np.any(mask_candidates):
        # We only dot-product the specific rows that passed the check
        # block_ins[mask_candidates] extracts only relevant rows
        valid_ins_block = block_ins[mask_candidates]
        
        # Calculate Insert dots
        dots_ins = valid_ins_block.dot(row_ins.T) # This handles dense/sparse correctly
        
        # If dots_ins is a matrix (happens if svd output is dense), flatten it
        if hasattr(dots_ins, "toarray"):
            dots_ins = dots_ins.toarray().flatten()
        else:
            dots_ins = np.ravel(dots_ins)
            
        dists_ins = 1.0 - dots_ins
        dists_ins[dists_ins < 0] = 0
        
        # 5. STRICT FUSION (Max)
        # Combine BC and Ins distances only for the valid ones
        # We grab the BC dists for these specific ones
        current_bc_dists = dists_bc[mask_candidates]
        
        # Apply your logic: max(bc, ins)
        # Note: We removed rank_normalization because it requires the full N^2 matrix.
        # Raw cosine distance is stable enough here.
        fused_vals = np.maximum(current_bc_dists, dists_ins)
        
        # Write back to the fused array
        fused_dists[mask_candidates] = fused_vals

    # 6. Fill the condensed matrix
    # The block size matches exactly the remaining slots for this row
    n_block = len(fused_dists)
    condensed_matrix[k_counter : k_counter + n_block] = fused_dists
    k_counter += n_block

print("    Fusion Complete.")


# --- 4. CLUSTERING (Unchanged) ---
print(f"[3/3] Clustering...")

Z = linkage(condensed_matrix, method='average')

# Stability Analysis 
merge_distances = Z[:, 2]
n_samples = len(df)
target_k = n_entities # Approx target
window = 200

best_k = target_k
max_lifetime = -1.0
best_threshold = 0.0

start_k = max(2, target_k - window)
end_k = min(n_samples - 1, target_k + window)

for k in range(start_k, end_k):
    idx = n_samples - k
    if idx >= len(merge_distances): continue
    
    lifetime = merge_distances[idx] - merge_distances[idx-1]
    
    if lifetime > max_lifetime:
        max_lifetime = lifetime
        best_k = k
        best_threshold = merge_distances[idx-1] + (lifetime / 2)

print(f"    Stability Optimized K: {best_k} | Threshold: {best_threshold:.4f}")

labels = fcluster(Z, t=best_threshold, criterion='distance')
df['Cluster'] = labels


# --- 5. VALIDATION (Unchanged) ---
print("\n--- Final Forensic Report ---")

t1 = df[df['Label'].str.contains("Trap_SharedBC")]
t1_clusters = t1['Cluster'].unique()
print(f"Trap 1 (Shared BC, Diff Ins): Found in {len(t1_clusters)} clusters.")
if len(t1_clusters) == 2: print("    ✅ SUCCESS")
else: print("    ❌ FAIL")

t2 = df[df['Label'].str.contains("Trap_SharedIns")]
t2_clusters = t2['Cluster'].unique()
print(f"Trap 2 (Diff BC, Shared Ins): Found in {len(t2_clusters)} clusters.")
if len(t2_clusters) == 2: print("    ✅ SUCCESS")
else: print("    ❌ FAIL")

n_final = len(set(df['Cluster']))
print(f"\nTotal Clusters: {n_final} (True: {len(combos)})")

--- Generating Simulated Data (Targeting Scalability) ---
Simulating 2000 entities...
Dataset Generated: 10030 reads.

[1/3] Vectorizing Sequences...
[2/3] Fusing Signals (Sparse/Gated Calculation)...
    Target condensed size: 50,295,435 elements.
    Processing Row 0/10030...
    Processing Row 1003/10030...
    Processing Row 2006/10030...
    Processing Row 3009/10030...
    Processing Row 4012/10030...
    Processing Row 5015/10030...
    Processing Row 6018/10030...
    Processing Row 7021/10030...
    Processing Row 8024/10030...
    Processing Row 9027/10030...
    Fusion Complete.
[3/3] Clustering...
    Stability Optimized K: 2001 | Threshold: 0.3456

--- Final Forensic Report ---
Trap 1 (Shared BC, Diff Ins): Found in 2 clusters.
    ✅ SUCCESS
Trap 2 (Diff BC, Shared Ins): Found in 2 clusters.
    ✅ SUCCESS

Total Clusters: 2001 (True: 2001)


In [23]:
import numpy as np
import pandas as pd
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

# --- 1. DATA GENERATION (Targeting 5,000 sequences now) ---
print("--- Generating 5,000 Sequences ---")
random.seed(42)
np.random.seed(42)

def mutate(seq):
    s = list(seq)
    if random.random() < 0.95: return seq 
    idx = random.randint(0, len(s)-1)
    s[idx] = "N"
    return "".join(s)

def gen_dna(k): return "".join(random.choices("ACGT", k=k))

n_items = 20000 
pool_bc = [gen_dna(20) for _ in range(n_items)]
pool_ins = [gen_dna(150) for _ in range(n_items)]
data = []
# Generate reads (simplified for speed)
for i in range(n_entities):
    # Create a cluster of size 2-4
    n_reads = random.randint(2, 4)
    for _ in range(n_reads):
        data.append({
            "ID": i, # True Label
            "Barcode": mutate(pool_bc[i]),
            "Insert": mutate(pool_ins[i])
        })

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} reads")


# --- 2. VECTORIZATION (Unchanged) ---
print("\n[1/3] Vectorizing...")
# Barcode: Positional K-mers (3,5)
bc_vec = CountVectorizer(analyzer='char', ngram_range=(3, 5), binary=False, dtype=np.float32)
bc_mat = bc_vec.fit_transform(df['Barcode'])
bc_mat = Normalizer(norm='l2').transform(bc_mat)

# Insert: SVD
ins_vec = CountVectorizer(analyzer='char', ngram_range=(6, 6), binary=False, dtype=np.float32)
ins_raw = ins_vec.fit_transform(df['Insert'])
svd = TruncatedSVD(n_components=30, random_state=42) # Lower components for speed
ins_mat = Normalizer(norm='l2').transform(svd.fit_transform(ins_raw))


# --- 3. GREEDY CLUSTER EXTRACTION (Your Proposal) ---
print("\n[2/3] Running Greedy Extraction Loop...")

# State tracking
n_samples = len(df)
unassigned_mask = np.ones(n_samples, dtype=bool) # True = Available
cluster_labels = np.full(n_samples, -1, dtype=int)
current_cluster_id = 0

# Thresholds
# Coarse Gate: Barcode must be fairly close to even be considered
BC_THRESHOLD = 0.5 
# Strict Limit: Combined distance must be tight to be accepted
FINAL_THRESHOLD = 0.25 

# Loop until everyone is assigned
while np.any(unassigned_mask):
    
    # 1. Pick a RANDOM SEED from the unassigned pool
    # Get indices of all True values in the mask
    available_indices = np.flatnonzero(unassigned_mask)
    
    # Random choice
    seed_idx = np.random.choice(available_indices)
    
    # 2. Get the Seed's Vectors
    seed_bc = bc_mat[seed_idx]
    seed_ins = ins_mat[seed_idx]
    
    # 3. Calculate Barcode Distances vs ALL UNASSIGNED
    # (We operate on the subset to save compute, though sparse logic handles full matrix well too)
    # Optimization: Dot product against the whole sparse matrix is extremely fast
    # We filter by the mask later
    
    # Note: If memory is tight, you can slice bc_mat[unassigned_mask], 
    # but index mapping gets annoying. Easier to dot all and mask result.
    all_bc_dots = bc_mat.dot(seed_bc.T).toarray().flatten()
    all_bc_dists = 1.0 - all_bc_dots
    
    # 4. Find Candidates (Limit to Top 1000 Closest)
    MAX_CANDIDATES = 1000
    
    # Identify indices that are Unassigned AND pass the Coarse Threshold
    # (Using flatnonzero is faster than boolean masking for the subsequent sort)
    potential_indices = np.flatnonzero(unassigned_mask & (all_bc_dists < BC_THRESHOLD))
    
    # If we have too many candidates, pick only the closest 1000
    if len(potential_indices) > MAX_CANDIDATES:
        # Get the actual distances for these potential candidates
        potential_dists = all_bc_dists[potential_indices]
        
        # argpartition puts the smallest K elements first (unsorted) - extremely fast
        # We find the local indices of the top 1000 best matches
        top_local_idx = np.argpartition(potential_dists, MAX_CANDIDATES)[:MAX_CANDIDATES]
        
        # Map back to the global dataframe indices
        candidate_indices = potential_indices[top_local_idx]
    else:
        # If fewer than 1000, take them all
        candidate_indices = potential_indices
    
    # 5. strict Validation (Check Inserts)
    # Only calculate Insert Distances for the Candidates
    if len(candidate_indices) > 0:

        # print(len(candidate_indices))
        
        # Get Insert Vectors for candidates
        cand_ins_vecs = ins_mat[candidate_indices]
        
        # Calculate Insert Distances vs Seed
        cand_ins_dots = cand_ins_vecs.dot(seed_ins.T)
        if hasattr(cand_ins_dots, "toarray"): 
            cand_ins_dots = cand_ins_dots.toarray().flatten()
        
        cand_ins_dists = 1.0 - cand_ins_dots
        
        # Fusion Logic (Max Rule)
        cand_bc_dists = all_bc_dists[candidate_indices]
        final_dists = np.maximum(cand_bc_dists, cand_ins_dists)
        
        # 6. Final Selection
        # Which ones passed the strict limit?
        accepted_mask = final_dists < FINAL_THRESHOLD
        accepted_indices = candidate_indices[accepted_mask]

        # print(f"{len(candidate_indices)} to {sum(accepted_mask)}")
        
        # Assign Cluster ID
        cluster_labels[accepted_indices] = current_cluster_id
        
        # Remove from pool
        unassigned_mask[accepted_indices] = False
        
        current_cluster_id += 1
        
    # Periodic Status Update
    if current_cluster_id % 500 == 0:
        remaining = np.sum(unassigned_mask)
        print(f"    Clusters Found: {current_cluster_id} | Remaining Reads: {remaining}")

df['Cluster'] = cluster_labels
print(f"Extraction Complete. Total Clusters: {current_cluster_id}")


# --- 4. VALIDATION ---
print("\n[3/3] Checking Accuracy...")

# We check "Cluster Purity"
# For each found cluster, how many unique TRUE IDs are inside?
# Ideally, exactly 1.
import collections
purity_scores = []

# Check first 1000 clusters to save time
for cid in range(min(1000, current_cluster_id)):
    cluster_subset = df[df['Cluster'] == cid]
    unique_true_ids = cluster_subset['ID'].unique()
    
    # Calculate Purity: (Most Common ID Count) / (Total Reads in Cluster)
    # If a cluster has 4 reads, and they are all ID 100 -> Purity 1.0
    # If a cluster has reads from ID 100 and ID 101 mixed -> Purity < 1.0
    if len(cluster_subset) > 0:
        counts = cluster_subset['ID'].value_counts(normalize=True)
        purity_scores.append(counts.iloc[0])

avg_purity = np.mean(purity_scores)
print(f"Average Cluster Purity: {avg_purity*100:.2f}%")

--- Generating 5,000 Sequences ---
Dataset: 15005 reads

[1/3] Vectorizing...

[2/3] Running Greedy Extraction Loop...
    Clusters Found: 500 | Remaining Reads: 13381
    Clusters Found: 1000 | Remaining Reads: 11794
    Clusters Found: 1500 | Remaining Reads: 10237
    Clusters Found: 2000 | Remaining Reads: 8680
    Clusters Found: 2500 | Remaining Reads: 7140
    Clusters Found: 3000 | Remaining Reads: 5589
    Clusters Found: 3500 | Remaining Reads: 4096
    Clusters Found: 4000 | Remaining Reads: 2641
    Clusters Found: 4500 | Remaining Reads: 1262
    Clusters Found: 5000 | Remaining Reads: 35
Extraction Complete. Total Clusters: 5018

[3/3] Checking Accuracy...
Average Cluster Purity: 100.00%
